# Encoding Methods Test

이 노트북은 `Byte2RGB` 클래스에 구현된 모든 인코딩 방식을 테스트합니다.

## 지원 인코딩 방식

| 인코딩 | 픽셀/바이트 | 오류 수정 능력 | 설명 |
|--------|------------|--------------|------|
| `legacy-bin` | 1 | 없음 | RGBBinning 직접 매핑 |
| `cube-id` | 1 | 없음 | RGB 큐브 ID 매핑 |
| `golay24` | 24 | 최대 3비트 오류 | Extended Golay(24,12) |
| `linear48` | 2 | 최대 8비트 오류 | Binary linear [48,8,17] ECC |
| `golay24-dual` | 2 | 최대 6비트 오류 | 2× Extended Golay(24,12) |
| `rs48` | 2 | 최대 2바이트 오류 | Reed-Solomon RS(6,1)/GF(2^8) |
| `bch48` | 2 | 최대 7비트 오류 | BCH[63,24,15] shortened |

## 0. 환경 설정

In [ ]:
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists():
            return path
    return start

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import random
import copy
from typing import List, Tuple

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

from diffusion_hash_inv.config import Byte2RGBConfig, HashConfig, MainConfig
from diffusion_hash_inv.core import RGB
from diffusion_hash_inv.utils.byte2rgb import Byte2RGB
from diffusion_hash_inv.utils.ecc48 import SUPPORTED_METHODS, get_codec, DecodeResult
from diffusion_hash_inv.validation.encoding_validation import encoding_validate

print("Import 완료")
print(f"48-bit ECC 지원 메서드: {SUPPORTED_METHODS}")

In [ ]:
# 공통 설정
MAIN_CFG = MainConfig(
    verbose_flag=False,
    clean_flag=False,
    debug_flag=False,
    make_image_flag=False,
)
HASH_CFG = HashConfig(hash_alg="md5", length=16)

ALL_ENCODINGS = ["legacy-bin", "cube-id", "golay24", "linear48", "golay24-dual", "rs48", "bch48"]

def make_encoder(encoding: str) -> Byte2RGB:
    cfg = Byte2RGBConfig(encoding=encoding, seed_flag=False, input_seed=42)
    return Byte2RGB(MAIN_CFG, HASH_CFG, cfg)

encoders = {enc: make_encoder(enc) for enc in ALL_ENCODINGS}
print(f"인코더 {len(encoders)}개 생성 완료: {list(encoders.keys())}")

## 1. 단일 바이트 인코딩/디코딩 라운드트립 테스트

각 인코딩 방식에 대해 단일 바이트 값을 인코딩하고 원래 값으로 복원되는지 확인합니다.

In [ ]:
TEST_BYTES = [b"\x00", b"\xff", b"\xab", b"\x42", b"\x89"]

print(f"{'인코딩':<15} {'입력':>8} {'출력 픽셀수':>12} {'디코딩 결과':>15} {'일치':>6}")
print("-" * 60)

for enc_name, encoder in encoders.items():
    for test_byte in TEST_BYTES[:3]:  # 처음 3개만 출력
        encoded = encoder.rgb_encoder(test_byte)
        pixels = encoded if isinstance(encoded, tuple) else (encoded,)
        decoded = encoder.rgb_decoder(pixels)
        match = "✓" if decoded == test_byte else "✗"
        print(f"{enc_name:<15} {test_byte.hex():>8} {len(pixels):>12} {decoded.hex():>15} {match:>6}")

## 2. 전체 256바이트 라운드트립 테스트

모든 가능한 바이트 값(0x00 ~ 0xFF)에 대해 인코딩 → 디코딩이 정확한지 검증합니다.

In [ ]:
results = {}

for enc_name, encoder in encoders.items():
    passed = 0
    failed_values = []
    for value in range(256):
        test_byte = bytes([value])
        encoded = encoder.rgb_encoder(test_byte)
        pixels = encoded if isinstance(encoded, tuple) else (encoded,)
        decoded = encoder.rgb_decoder(pixels)
        if decoded == test_byte:
            passed += 1
        else:
            failed_values.append(value)
    results[enc_name] = {"passed": passed, "failed": failed_values}

print(f"{'인코딩':<15} {'통과':>8} {'실패':>8} {'성공률':>10}")
print("-" * 45)
for enc_name, res in results.items():
    rate = res["passed"] / 256 * 100
    status = "✓" if res["passed"] == 256 else "✗"
    failed_str = str(res["failed"][:5]) if res["failed"] else ""
    print(f"{enc_name:<15} {res['passed']:>8} {len(res['failed']):>8} {rate:>9.1f}% {status}  {failed_str}")

## 3. 픽셀/바이트 비율 확인

각 인코딩 방식이 1바이트를 표현하는 데 사용하는 RGB 픽셀 수를 확인합니다.

In [ ]:
print(f"{'인코딩':<15} {'픽셀/바이트':>12} {'4바이트 예시 픽셀수':>20}")
print("-" * 50)

for enc_name, encoder in encoders.items():
    ppb = encoder.pixels_per_byte
    test_4bytes = b"\x01\x02\x03\x04"
    encoded = encoder.rgb_encoder(test_4bytes)
    pixel_count = len(encoded) if isinstance(encoded, tuple) else 1
    print(f"{enc_name:<15} {ppb:>12} {pixel_count:>20}")

## 4. 다중 바이트 인코딩 테스트

여러 바이트 시퀀스를 인코딩하고 정확히 복원되는지 테스트합니다.

In [ ]:
MULTI_BYTE_TESTS = [
    b"\x89\xab\xcd\xef",
    b"Hello",
    bytes(range(16)),  # 0x00 ~ 0x0f
]

for test_data in MULTI_BYTE_TESTS:
    print(f"\n입력 ({len(test_data)} bytes): {test_data.hex()}")
    print(f"  {'인코딩':<15} {'픽셀수':>8} {'복원 성공':>10}")
    for enc_name, encoder in encoders.items():
        encoded = encoder.rgb_encoder(test_data)
        pixels = encoded if isinstance(encoded, tuple) else (encoded,)
        decoded = encoder.rgb_decoder(pixels)
        ok = "✓" if decoded == test_data else f"✗ ({decoded.hex()})"
        print(f"  {enc_name:<15} {len(pixels):>8} {ok:>10}")

## 5. ECC 오류 수정 능력 테스트

48-bit ECC 인코딩 방식에 인위적으로 비트 오류를 주입하고 복원 가능한지 확인합니다.

In [ ]:
def flip_bit_in_codeword(codeword: bytes, bit_index: int) -> bytes:
    b = bytearray(codeword)
    byte_pos = bit_index // 8
    bit_pos = 7 - (bit_index % 8)
    b[byte_pos] ^= (1 << bit_pos)
    return bytes(b)

# 48-bit ECC 인코딩: 1바이트 → 2픽셀(48bit)
ECC_ENCODINGS = {
    "linear48": 8,      # 최대 수정 가능 비트 수
    "golay24-dual": 6,
    "rs48": None,       # 바이트 오류 기준
    "bch48": 7,
}

TEST_VALUE = 0xA5
test_byte = bytes([TEST_VALUE])

print(f"테스트 바이트: 0x{TEST_VALUE:02X}\n")

for enc_name, max_correctable in ECC_ENCODINGS.items():
    if enc_name == "rs48":
        continue  # RS48는 바이트 오류 기준으로 별도 테스트
    
    codec = get_codec(enc_name)
    codeword = codec.encode(TEST_VALUE)
    
    print(f"[{enc_name}] 최대 수정 비트: {max_correctable}")
    print(f"  {'오류 비트 수':>12} {'수정 성공':>12} {'confidence':>12}")
    
    for num_errors in range(0, max_correctable + 2):
        corrupted = codeword
        for i in range(num_errors):
            corrupted = flip_bit_in_codeword(corrupted, i * 6)  # 고르게 분포
        
        result = codec.decode(corrupted)
        success = "✓" if result.valid and result.payload == TEST_VALUE else "✗"
        limit_note = " ← 한계 초과" if num_errors > max_correctable else ""
        print(f"  {num_errors:>12} {success:>12} {result.confidence:>12.3f}{limit_note}")
    print()

### 5-1. RS48 바이트 오류 수정 테스트

In [ ]:
from diffusion_hash_inv.utils.ecc48 import RS48Codec

codec_rs = RS48Codec()
codeword = codec_rs.encode(TEST_VALUE)  # 6 bytes

print(f"[rs48] 바이트 오류 수정 테스트 (최대 2바이트)")
print(f"원본 codeword: {codeword.hex()}")
print(f"  {'오류 바이트 수':>14} {'수정 성공':>12} {'confidence':>12}")

for num_byte_errors in range(0, 4):
    corrupted = bytearray(codeword)
    for i in range(num_byte_errors):
        corrupted[i] ^= 0xFF
    
    result = codec_rs.decode(bytes(corrupted))
    success = "✓" if result.valid and result.payload == TEST_VALUE else "✗"
    limit_note = " ← 한계 초과" if num_byte_errors > 2 else ""
    print(f"  {num_byte_errors:>14} {success:>12} {result.confidence:>12.3f}{limit_note}")

### 5-2. Golay24 (24픽셀/바이트) 오류 수정 테스트

In [ ]:
encoder_g24 = encoders["golay24"]

test_byte = bytes([0x12])
encoded = list(encoder_g24.rgb_encoder(test_byte))  # 24 RGB pixels

print(f"[golay24] 최대 3비트 오류 수정 테스트")
print(f"인코딩된 픽셀 수: {len(encoded)}")
print(f"  {'오류 비트 수':>12} {'수정 성공':>12}")

for num_errors in range(0, 5):
    corrupted = list(encoded)
    # 픽셀의 octant 값을 변경해서 오류 주입
    for i in range(num_errors):
        idx = i * 8  # 각 바이트 위치에서 오류
        if idx < len(corrupted):
            bit_pos = idx % encoder_g24.data_bits_per_byte
            octant = encoder_g24.rgb_octant_decoder(corrupted[idx])
            wrong_octant = bit_pos if octant != bit_pos else bit_pos ^ 0b111
            corrupted[idx] = encoder_g24._rgb_from_octant(wrong_octant)
    
    decoded = encoder_g24.rgb_decoder(tuple(corrupted))
    success = "✓" if decoded == test_byte else "✗"
    limit_note = " ← 한계 초과" if num_errors > 3 else ""
    print(f"  {num_errors:>12} {success:>12}{limit_note}")

## 6. encoding_validate() 검증 함수 테스트

`validation` 모듈의 `encoding_validate()` 함수를 이용하여 인코딩 검증을 수행합니다.

In [ ]:
# verbose_flag=True 인코더로 검증 함수 테스트
verbose_main_cfg = MainConfig(
    verbose_flag=True,
    clean_flag=False,
    debug_flag=False,
    make_image_flag=False,
)

print("=== encoding_validate() 테스트 ===")
for enc_name in ["linear48", "cube-id", "legacy-bin"]:
    cfg = Byte2RGBConfig(encoding=enc_name, seed_flag=False, input_seed=42)
    encoder = Byte2RGB(verbose_main_cfg, HASH_CFG, cfg)
    
    test_data = b"\xde\xad"
    encoded = encoder.rgb_encoder(test_data)
    
    print(f"\n[{enc_name}] 입력: {test_data.hex()}")
    result = encoding_validate(test_data, encoded, encoder)
    print(f"검증 결과: {'통과' if result else '실패'}")

## 7. 인코딩 결과 시각화

각 인코딩 방식으로 `0xFF`를 인코딩했을 때 생성되는 RGB 픽셀을 시각화합니다.

In [ ]:
test_byte_vis = bytes([0xFF])

# golay24는 픽셀이 너무 많으므로 처음 8개만 표시
MAX_PIXELS_SHOWN = 8

fig, axes = plt.subplots(len(ALL_ENCODINGS), 1, figsize=(14, len(ALL_ENCODINGS) * 1.4))
fig.suptitle(f"인코딩 방식별 RGB 픽셀 시각화 (입력: 0x{test_byte_vis.hex().upper()})", fontsize=13)

for ax, enc_name in zip(axes, ALL_ENCODINGS):
    encoder = encoders[enc_name]
    encoded = encoder.rgb_encoder(test_byte_vis)
    pixels = encoded if isinstance(encoded, tuple) else (encoded,)
    
    display_pixels = pixels[:MAX_PIXELS_SHOWN]
    n = len(display_pixels)
    
    for i, pixel in enumerate(display_pixels):
        color = [pixel.r / 255, pixel.g / 255, pixel.b / 255]
        rect = patches.FancyBboxPatch(
            (i, 0), 0.9, 0.9,
            boxstyle="round,pad=0.05",
            facecolor=color,
            edgecolor="gray",
            linewidth=0.5
        )
        ax.add_patch(rect)
        # 픽셀 값 텍스트 (밝기에 따라 색상 선택)
        brightness = 0.299 * pixel.r + 0.587 * pixel.g + 0.114 * pixel.b
        txt_color = "black" if brightness > 128 else "white"
        ax.text(i + 0.45, 0.45, f"({pixel.r},{pixel.g},{pixel.b})",
                ha="center", va="center", fontsize=6.5, color=txt_color)
    
    if len(pixels) > MAX_PIXELS_SHOWN:
        ax.text(n + 0.1, 0.45, f"... +{len(pixels) - MAX_PIXELS_SHOWN}px",
                va="center", fontsize=8, color="gray")
    
    ax.set_xlim(-0.2, MAX_PIXELS_SHOWN + 1.5)
    ax.set_ylim(-0.1, 1.0)
    ax.set_yticks([])
    ax.set_xticks([])
    ax.set_ylabel(enc_name, fontsize=9, rotation=0, labelpad=85, va="center")
    ax.spines[:].set_visible(False)

plt.tight_layout()
plt.show()

## 8. RGB 공간에서 인코딩 분포 시각화

모든 256가지 바이트 값을 인코딩했을 때 RGB 공간에서의 픽셀 분포를 3D 산점도로 표시합니다.

In [ ]:
# 1픽셀/바이트 방식만 시각화 (legacy-bin, cube-id)
ONE_PIXEL_ENCODINGS = ["legacy-bin", "cube-id"]

fig = plt.figure(figsize=(14, 6))

for plot_idx, enc_name in enumerate(ONE_PIXEL_ENCODINGS, 1):
    encoder = encoders[enc_name]
    
    r_vals, g_vals, b_vals, colors = [], [], [], []
    for val in range(256):
        pixel = encoder.rgb_encoder(bytes([val]))
        if isinstance(pixel, tuple):
            pixel = pixel[0]
        r_vals.append(pixel.r)
        g_vals.append(pixel.g)
        b_vals.append(pixel.b)
        colors.append([pixel.r / 255, pixel.g / 255, pixel.b / 255])
    
    ax = fig.add_subplot(1, 2, plot_idx, projection="3d")
    ax.scatter(r_vals, g_vals, b_vals, c=colors, s=40, alpha=0.8)
    ax.set_title(f"{enc_name}\n256가지 바이트 RGB 분포", fontsize=11)
    ax.set_xlabel("R", fontsize=9)
    ax.set_ylabel("G", fontsize=9)
    ax.set_zlabel("B", fontsize=9)
    ax.set_xlim(0, 255)
    ax.set_ylim(0, 255)
    ax.set_zlim(0, 255)

plt.tight_layout()
plt.show()

In [ ]:
# 48-bit ECC 방식: 각 바이트의 2개 픽셀 첫 번째 픽셀 분포 시각화
ECC48_ENCODINGS = ["linear48", "golay24-dual", "rs48", "bch48"]

fig = plt.figure(figsize=(14, 12))

for plot_idx, enc_name in enumerate(ECC48_ENCODINGS, 1):
    encoder = encoders[enc_name]
    
    r1_vals, g1_vals, b1_vals = [], [], []
    r2_vals, g2_vals, b2_vals = [], [], []
    colors1, colors2 = [], []
    
    for val in range(256):
        pixels = encoder.rgb_encoder(bytes([val]))
        if isinstance(pixels, tuple) and len(pixels) >= 2:
            p1, p2 = pixels[0], pixels[1]
        else:
            continue
        r1_vals.append(p1.r); g1_vals.append(p1.g); b1_vals.append(p1.b)
        r2_vals.append(p2.r); g2_vals.append(p2.g); b2_vals.append(p2.b)
        colors1.append([p1.r/255, p1.g/255, p1.b/255])
        colors2.append([p2.r/255, p2.g/255, p2.b/255])
    
    ax = fig.add_subplot(2, 2, plot_idx, projection="3d")
    ax.scatter(r1_vals, g1_vals, b1_vals, c=colors1, s=30, alpha=0.7, label="RGB1", marker="o")
    ax.scatter(r2_vals, g2_vals, b2_vals, c=colors2, s=30, alpha=0.5, label="RGB2", marker="^")
    ax.set_title(f"{enc_name}\n256바이트 (○=RGB1, △=RGB2)", fontsize=10)
    ax.set_xlabel("R", fontsize=8); ax.set_ylabel("G", fontsize=8); ax.set_zlabel("B", fontsize=8)
    ax.set_xlim(0, 255); ax.set_ylim(0, 255); ax.set_zlim(0, 255)

plt.tight_layout()
plt.show()

## 9. DecodeResult 상세 정보 확인

48-bit ECC 코덱의 `DecodeResult` 에 포함된 신뢰도(confidence) 및 오류 수정 정보를 확인합니다.

In [ ]:
SAMPLE_VALUES = [0x00, 0x7F, 0xFF, 0xA5]

for enc_name in SUPPORTED_METHODS:
    encoder = encoders[enc_name]
    print(f"\n[{enc_name}]")
    print(f"  {'바이트':>8} {'payload':>10} {'confidence':>12} {'errors_corrected':>18} {'valid':>7}")
    
    for val in SAMPLE_VALUES:
        test_b = bytes([val])
        encoded = encoder.rgb_encoder(test_b)
        pixels = encoded if isinstance(encoded, tuple) else (encoded,)
        result: DecodeResult = encoder.decode_payload_with_confidence(pixels[0], pixels[1])
        print(f"  {f'0x{val:02X}':>8} {str(result.payload):>10} {result.confidence:>12.3f} "
              f"{result.errors_corrected:>18} {str(result.valid):>7}")

## 10. 인코딩 방식 비교 요약

In [ ]:
import textwrap

summary_data = {
    "legacy-bin":    {"px/byte": 1,  "ecc": "없음",         "정확도": "100%", "비고": "RGBBinning 직접 매핑"},
    "cube-id":       {"px/byte": 1,  "ecc": "없음",         "정확도": "100%", "비고": "RGB 큐브 ID 공식"},
    "golay24":       {"px/byte": 24, "ecc": "≤3 비트",      "정확도": "100%", "비고": "Ext. Golay(24,12)"},
    "linear48":      {"px/byte": 2,  "ecc": "≤8 비트",      "정확도": "100%", "비고": "Binary [48,8,17]"},
    "golay24-dual":  {"px/byte": 2,  "ecc": "≤6 비트",      "정확도": "100%", "비고": "2× Golay(24,12)"},
    "rs48":          {"px/byte": 2,  "ecc": "≤2 바이트",    "정확도": "100%", "비고": "RS(6,1)/GF(2^8)"},
    "bch48":         {"px/byte": 2,  "ecc": "≤7 비트",      "정확도": "100%", "비고": "BCH[63,24,15]"},
}

# 실제 테스트 결과로 정확도 업데이트
for enc_name, res in results.items():
    rate = res["passed"] / 256 * 100
    summary_data[enc_name]["정확도"] = f"{rate:.0f}% ({res['passed']}/256)"

header = f"{'인코딩':<15} {'px/byte':>8} {'ECC 능력':>12} {'256 라운드트립':>18} {'비고'}"
print(header)
print("-" * 80)
for enc_name, d in summary_data.items():
    print(f"{enc_name:<15} {d['px/byte']:>8} {d['ecc']:>12} {d['정확도']:>18}  {d['비고']}")

print("\n✓ = 256/256 라운드트립 성공")

## 11. 픽셀 효율성 vs. 오류 수정 능력 시각화

In [ ]:
enc_labels  = ["legacy-bin", "cube-id", "golay24", "linear48", "golay24-dual", "rs48", "bch48"]
px_per_byte = [1, 1, 24, 2, 2, 2, 2]
ecc_bits    = [0, 0, 3, 8, 6, 16, 7]  # rs48: 2 bytes ≈ 16 bits
colors_bar  = ["#aaaaaa", "#888888", "#5588cc", "#cc4444", "#ee8833", "#44aa44", "#aa44cc"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽: 픽셀 수
bars1 = ax1.bar(enc_labels, px_per_byte, color=colors_bar, edgecolor="white", linewidth=0.7)
ax1.set_title("인코딩 방식별 픽셀/바이트", fontsize=12)
ax1.set_ylabel("픽셀 수 (로그 스케일)")
ax1.set_yscale("log")
ax1.set_ylim(0.5, 50)
for bar, val in zip(bars1, px_per_byte):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.1,
             str(val), ha="center", va="bottom", fontsize=10, fontweight="bold")
ax1.tick_params(axis="x", rotation=30)
ax1.grid(axis="y", alpha=0.3)

# 오른쪽: ECC 능력
bars2 = ax2.bar(enc_labels, ecc_bits, color=colors_bar, edgecolor="white", linewidth=0.7)
ax2.set_title("인코딩 방식별 ECC 수정 능력 (비트 환산)", fontsize=12)
ax2.set_ylabel("수정 가능 비트 수")
for bar, val, label in zip(bars2, ecc_bits, enc_labels):
    note = "(2 bytes)" if label == "rs48" else ""
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
             f"{val}{note}", ha="center", va="bottom", fontsize=9)
ax2.tick_params(axis="x", rotation=30)
ax2.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()